In [1]:
import os,sys
import pyarrow as pa

In [2]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from itertools import combinations
%pip install rapidfuzz
from rapidfuzz.fuzz import ratio, token_set_ratio, token_sort_ratio

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "student_resource"/ "dataset" / "test"
RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR = PROJECT_ROOT / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

Note: you may need to restart the kernel to use updated packages.
Project root: D:\V-Max_Hackerz


In [3]:
s1_test = pd.read_csv(
    DATA_DIR / "test_source1.tsv",
    sep="\t",
    dtype=str
)

s2_test = pd.read_csv(
    DATA_DIR / "test_source2.tsv",
    sep="\t",
    dtype=str
)

s3_test = pd.read_csv(
    DATA_DIR / "test_source3.tsv",
    sep="\t",
    dtype=str
)

candidate_pairs = pd.read_csv(
    OUTPUT_DIR / "candidate_pairs.tsv",
    sep="\t",
    dtype=str
)

print("S1 test:", s1_test.shape)
print("S2 test:", s2_test.shape)
print("S3 test:", s3_test.shape)
print("Candidate pairs:", candidate_pairs.shape)

S1 test: (1732544, 4)
S2 test: (4887273, 4)
S3 test: (5082316, 4)
Candidate pairs: (1732544, 2)


In [ ]:
!pip install xgboost
import xgboost as xgb
print(xgb.__version__)
print(xgb.__file__)

In [6]:
model=xgb.XGBClassifier()
model.load_model(
    RESULTS_DIR / "xgboost_entity_resolution.json"
)


threshold = float(
    (RESULTS_DIR / "decision_threshold.txt").read_text().strip()
)

print("Model loaded:", type(model).__name__)
print("Threshold:", threshold)

Model loaded: XGBClassifier
Threshold: 0.67


In [4]:
candidate_source_test = pd.concat(
    [
        s2_test[
            ["entity_id", "business_name", "business_address", "country"]
        ],
        s3_test[
            ["entity_id", "business_name", "business_address", "country"]
        ]
    ],
    ignore_index=True
).set_index("entity_id")

print("Test candidate entities:", len(candidate_source_test))

Test candidate entities: 9969589


In [5]:
s1_test_lookup = (
    s1_test
    .set_index("entity_id")
    [["business_name", "business_address", "country"]]
    .to_dict("index")
)

candidate_test_lookup = (
    candidate_source_test
    [["business_name", "business_address", "country"]]
    .to_dict("index")
)

print("S1 lookup:", len(s1_test_lookup))
print("Candidate lookup:", len(candidate_test_lookup))

S1 lookup: 1732544
Candidate lookup: 9969589


In [6]:
def normalize_unicode(text):
    if pd.isna(text):
        return ""
    return str(text).casefold()


def normalize_text(text):
    text = normalize_unicode(text)

    if not text:
        return ""

    text = text.replace("&", " and ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


NAME_ABBREVIATIONS = {
    "pvt": "private",
    "ltd": "limited",
    "corp": "corporation",
    "inc": "incorporated",
    "co": "company"
}

ADDRESS_ABBREVIATIONS = {
    "rd": "road",
    "st": "street",
    "ave": "avenue",
    "blvd": "boulevard",
    "dr": "drive",
    "ln": "lane",
    "hwy": "highway",
    "pkwy": "parkway",
    "ct": "court",
    "pl": "place",
    "trl": "trail",
    "cir": "circle",
    "ste": "suite",
    "apt": "apartment"
}


def expand_abbreviations(text, mapping):
    if not text:
        return ""

    return " ".join(
        mapping.get(token, token)
        for token in text.split()
    )


def normalize_name(text):
    return expand_abbreviations(
        normalize_text(text),
        NAME_ABBREVIATIONS
    )


def normalize_address(text):
    return expand_abbreviations(
        normalize_text(text),
        ADDRESS_ABBREVIATIONS
    )

In [7]:
def safe_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def token_jaccard(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0.0

    A = set(a.split())
    B = set(b.split())

    union = A | B

    return 0.0 if not union else len(A & B) / len(union)


def shared_token_count(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0

    return len(set(a.split()) & set(b.split()))


def length_ratio(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0.0

    return min(len(a), len(b)) / max(len(a), len(b))


def char_similarity(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0.0

    return ratio(a, b) / 100.0

In [8]:
import re
def make_features(s1_record, candidate_record):

    s1_name = safe_text(s1_record["business_name"])
    s2_name = safe_text(candidate_record["business_name"])

    s1_address = safe_text(s1_record["business_address"])
    s2_address = safe_text(candidate_record["business_address"])

    s1_country = safe_text(s1_record["country"])
    s2_country = safe_text(candidate_record["country"])

    norm_s1_name = normalize_name(s1_name)
    norm_s2_name = normalize_name(s2_name)

    norm_s1_address = normalize_address(s1_address)
    norm_s2_address = normalize_address(s2_address)

    return {
        "name_exact": int(
            norm_s1_name == norm_s2_name
            and norm_s1_name != ""
        ),

        "name_token_jaccard": token_jaccard(
            norm_s1_name, norm_s2_name
        ),

        "name_char_similarity": char_similarity(
            norm_s1_name, norm_s2_name
        ),

        "name_token_set_ratio": token_set_ratio(
            norm_s1_name, norm_s2_name
        ) / 100.0,

        "name_token_sort_ratio": token_sort_ratio(
            norm_s1_name, norm_s2_name
        ) / 100.0,

        "name_shared_tokens": shared_token_count(
            norm_s1_name, norm_s2_name
        ),

        "name_length_ratio": length_ratio(
            norm_s1_name, norm_s2_name
        ),

        "address_exact": int(
            norm_s1_address == norm_s2_address
            and norm_s1_address != ""
        ),

        "address_token_jaccard": token_jaccard(
            norm_s1_address, norm_s2_address
        ),

        "address_char_similarity": char_similarity(
            norm_s1_address, norm_s2_address
        ),

        "address_token_set_ratio": token_set_ratio(
            norm_s1_address, norm_s2_address
        ) / 100.0,

        "address_token_sort_ratio": token_sort_ratio(
            norm_s1_address, norm_s2_address
        ) / 100.0,

        "address_shared_tokens": shared_token_count(
            norm_s1_address, norm_s2_address
        ),

        "address_length_ratio": length_ratio(
            norm_s1_address, norm_s2_address
        ),

        "country_match": int(
            s1_country != "" and s1_country == s2_country
        ),

        "name_missing": int(
            not s1_name or not s2_name
        ),

        "address_missing": int(
            not s1_address or not s2_address
        ),
    }

In [14]:
CACHE_DIR = RESULTS_DIR / "inference_cache"
CHECKPOINT_DIR = RESULTS_DIR / "inference_checkpoints"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Cache:", CACHE_DIR)
print("Checkpoints:", CHECKPOINT_DIR)

Cache: D:\V-Max_Hackerz\results\inference_cache
Checkpoints: D:\V-Max_Hackerz\results\inference_checkpoints


In [13]:
from functools import lru_cache

@lru_cache(maxsize=1_000_000)
def cached_normalize_name(text):
    return normalize_name(text)

@lru_cache(maxsize=1_000_000)
def cached_normalize_address(text):
    return normalize_address(text)

In [18]:
!pip install pyarrow fastparquet
S1_CACHE = CACHE_DIR / "test_s1_features.parquet"

if S1_CACHE.exists():

    print("S1 cache found. Loading...")
    s1_cache = pd.read_parquet(S1_CACHE)

else:

    print("S1 cache not found. Creating...")

    s1_cache = s1_test[
        ["entity_id", "business_name", "business_address", "country"]
    ].copy()

    s1_cache["norm_name"] = (
        s1_cache["business_name"]
        .fillna("")
        .astype(str)
        .map(cached_normalize_name)
    )

    s1_cache["norm_address"] = (
        s1_cache["business_address"]
        .fillna("")
        .astype(str)
        .map(cached_normalize_address)
    )

    s1_cache["country_clean"] = (
        s1_cache["country"]
        .fillna("")
        .astype(str)
    )

    s1_cache["name_missing"] = (
        s1_cache["business_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .astype(np.int8)
    )

    s1_cache["address_missing"] = (
        s1_cache["business_address"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .astype(np.int8)
    )

    s1_cache.to_parquet(
        S1_CACHE,
        index=False
    )

    print("S1 cache created and saved.")

print("S1 cache shape:", s1_cache.shape)


S1 cache found. Loading...
S1 cache shape: (1732544, 9)


In [19]:
CANDIDATE_CACHE = CACHE_DIR / "test_candidate_features.parquet"

if CANDIDATE_CACHE.exists():

    print("Candidate cache found. Loading...")
    candidate_cache = pd.read_parquet(
        CANDIDATE_CACHE
    )

else:

    print("Candidate cache not found. Creating...")

    candidate_cache = candidate_source_test[
        ["business_name", "business_address", "country"]
    ].copy()

    candidate_cache["norm_name"] = (
        candidate_cache["business_name"]
        .fillna("")
        .astype(str)
        .map(cached_normalize_name)
    )

    candidate_cache["norm_address"] = (
        candidate_cache["business_address"]
        .fillna("")
        .astype(str)
        .map(cached_normalize_address)
    )

    candidate_cache["country_clean"] = (
        candidate_cache["country"]
        .fillna("")
        .astype(str)
    )

    candidate_cache["name_missing"] = (
        candidate_cache["business_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .astype(np.int8)
    )

    candidate_cache["address_missing"] = (
        candidate_cache["business_address"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .astype(np.int8)
    )

    candidate_cache.to_parquet(
        CANDIDATE_CACHE,
        index=True
    )

    print("Candidate cache created and saved.")

print("Candidate cache shape:", candidate_cache.shape)

Candidate cache found. Loading...
Candidate cache shape: (9969589, 8)


In [20]:
s1_cache = s1_cache.set_index(
    "entity_id",
    drop=False
)

candidate_cache = candidate_cache.sort_index()

print("S1 cache ready:", len(s1_cache))
print("Candidate cache ready:", len(candidate_cache))

S1 cache ready: 1732544
Candidate cache ready: 9969589


In [23]:
def build_feature_row_cached(s1_rec, candidate_rec):

    s1_name = s1_rec["norm_name"]
    s1_addr = s1_rec["norm_address"]
    s1_country = s1_rec["country_clean"]

    c_name = candidate_rec["norm_name"]
    c_addr = candidate_rec["norm_address"]
    c_country = candidate_rec["country_clean"]

    # Name tokens
    name_a = set(s1_name.split()) if s1_name else set()
    name_b = set(c_name.split()) if c_name else set()

    name_intersection = len(name_a & name_b)
    name_union = name_a | name_b

    name_jaccard = (
        name_intersection / len(name_union)
        if name_union else 0.0
    )

    # Address tokens
    addr_a = set(s1_addr.split()) if s1_addr else set()
    addr_b = set(c_addr.split()) if c_addr else set()

    addr_intersection = len(addr_a & addr_b)
    addr_union = addr_a | addr_b

    addr_jaccard = (
        addr_intersection / len(addr_union)
        if addr_union else 0.0
    )

    # Name similarities
    if s1_name and c_name:

        name_char = ratio(
            s1_name,
            c_name
        ) / 100.0

        name_set = token_set_ratio(
            s1_name,
            c_name
        ) / 100.0

        name_sort = token_sort_ratio(
            s1_name,
            c_name
        ) / 100.0

        name_len = (
            min(len(s1_name), len(c_name))
            / max(len(s1_name), len(c_name))
        )

    else:

        name_char = 0.0
        name_set = 0.0
        name_sort = 0.0
        name_len = 0.0

    # Address similarities
    if s1_addr and c_addr:

        addr_char = ratio(
            s1_addr,
            c_addr
        ) / 100.0

        addr_set = token_set_ratio(
            s1_addr,
            c_addr
        ) / 100.0

        addr_sort = token_sort_ratio(
            s1_addr,
            c_addr
        ) / 100.0

        addr_len = (
            min(len(s1_addr), len(c_addr))
            / max(len(s1_addr), len(c_addr))
        )

    else:

        addr_char = 0.0
        addr_set = 0.0
        addr_sort = 0.0
        addr_len = 0.0

    return [
        int(bool(s1_name) and s1_name == c_name),
        name_jaccard,
        name_char,
        name_set,
        name_sort,
        name_intersection,
        name_len,

        int(bool(s1_addr) and s1_addr == c_addr),
        addr_jaccard,
        addr_char,
        addr_set,
        addr_sort,
        addr_intersection,
        addr_len,

        int(
            bool(s1_country)
            and s1_country == c_country
        ),

        int(
            s1_rec["name_missing"]
            or candidate_rec["name_missing"]
        ),

        int(
            s1_rec["address_missing"]
            or candidate_rec["address_missing"]
        )
    ]

In [24]:
FEATURE_COLUMNS = [
    "name_exact",
    "name_token_jaccard",
    "name_char_similarity",
    "name_token_set_ratio",
    "name_token_sort_ratio",
    "name_shared_tokens",
    "name_length_ratio",

    "address_exact",
    "address_token_jaccard",
    "address_char_similarity",
    "address_token_set_ratio",
    "address_token_sort_ratio",
    "address_shared_tokens",
    "address_length_ratio",

    "country_match",
    "name_missing",
    "address_missing"
]

MATCH_THRESHOLD = 0.67

print("Features:", len(FEATURE_COLUMNS))
print("Threshold:", MATCH_THRESHOLD)


Features: 17
Threshold: 0.67


In [25]:
from rapidfuzz import process, fuzz
import numpy as np
import time

# Take the first 10,000 candidate pairs
benchmark_n = min(10_000, len(candidate_pairs))

names_a = []
names_b = []

for i in range(benchmark_n):

    row = candidate_pairs.iloc[i]

    s1_id = row.source1_entity_id
    candidate_string = row.candidate_entity_ids

    if pd.isna(candidate_string) or not candidate_string:
        continue

    s1_rec = s1_cache.loc[s1_id]

    candidate_ids = [
        x.strip()
        for x in candidate_string.split(",")
        if x.strip()
    ]

    for cid in candidate_ids:
        try:
            c = candidate_cache.loc[cid]
        except KeyError:
            continue

        names_a.append(s1_rec["norm_name"])
        names_b.append(c["norm_name"])

print("Pairs:", len(names_a))

start = time.time()

scores_ratio = process.cpdist(
    names_a,
    names_b,
    scorer=fuzz.ratio,
    workers=-1,
    dtype=np.float32
)

elapsed = time.time() - start

print(f"RapidFuzz time: {elapsed:.2f} seconds")
print("Scores:", scores_ratio.shape)
print("CPU workers: all available")

KeyboardInterrupt: 

In [3]:
from rapidfuzz import process, fuzz
def build_chunk_features_fast(candidate_pairs_chunk):
    """
    Build the exact same 17 features used during training,
    but calculate the expensive fuzzy features in bulk.
    """

    # --------------------------------------------------
    # 1. Flatten candidate_pairs into individual pairs
    # --------------------------------------------------

    s1_ids = []
    candidate_ids = []

    for row in candidate_pairs_chunk.itertuples(index=False):

        s1_id = row.source1_entity_id
        candidate_string = row.candidate_entity_ids

        if pd.isna(candidate_string) or not candidate_string:
            continue

        ids = [
            x.strip()
            for x in candidate_string.split(",")
            if x.strip()
        ]

        for cid in ids:
            s1_ids.append(s1_id)
            candidate_ids.append(cid)

    if not candidate_ids:
        return (
            np.empty((0, 17), dtype=np.float32),
            [],
            []
        )

    # --------------------------------------------------
    # 2. Batch lookup S1 records
    # --------------------------------------------------

    s1_frame = s1_cache.reindex(s1_ids)

    # --------------------------------------------------
    # 3. Batch lookup candidate records
    # --------------------------------------------------

    candidate_frame = candidate_cache.reindex(
        candidate_ids
    )

    # --------------------------------------------------
    # 4. Convert strings to arrays
    # --------------------------------------------------

    s1_names = (
        s1_frame["norm_name"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_names = (
        candidate_frame["norm_name"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    s1_addresses = (
        s1_frame["norm_address"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_addresses = (
        candidate_frame["norm_address"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    s1_countries = (
        s1_frame["country_clean"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_countries = (
        candidate_frame["country_clean"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    # --------------------------------------------------
    # 5. Exact matches
    # --------------------------------------------------

    name_exact = (
        (s1_names != "")
        & (s1_names == candidate_names)
    ).astype(np.float32)

    address_exact = (
        (s1_addresses != "")
        & (s1_addresses == candidate_addresses)
    ).astype(np.float32)

    country_match = (
        (s1_countries != "")
        & (s1_countries == candidate_countries)
    ).astype(np.float32)

    # --------------------------------------------------
    # 6. Missing flags
    # --------------------------------------------------

    name_missing = np.logical_or(
        s1_frame["name_missing"].to_numpy(dtype=np.int8),
        candidate_frame["name_missing"].to_numpy(dtype=np.int8)
    ).astype(np.float32)
        
    address_missing = np.logical_or(
        s1_frame["address_missing"].to_numpy(dtype=np.int8),
        candidate_frame["address_missing"].to_numpy(dtype=np.int8)
    ).astype(np.float32)

    # --------------------------------------------------
    # 7. String lengths
    # --------------------------------------------------

    s1_name_len = np.array(
        [len(x) for x in s1_names],
        dtype=np.float32
    )

    c_name_len = np.array(
        [len(x) for x in candidate_names],
        dtype=np.float32
    )

    name_length_ratio = np.divide(
        np.minimum(s1_name_len, c_name_len),
        np.maximum(s1_name_len, c_name_len),
        out=np.zeros_like(s1_name_len),
        where=np.maximum(s1_name_len, c_name_len) > 0
    )

    s1_addr_len = np.array(
        [len(x) for x in s1_addresses],
        dtype=np.float32
    )

    c_addr_len = np.array(
        [len(x) for x in candidate_addresses],
        dtype=np.float32
    )

    address_length_ratio = np.divide(
        np.minimum(s1_addr_len, c_addr_len),
        np.maximum(s1_addr_len, c_addr_len),
        out=np.zeros_like(s1_addr_len),
        where=np.maximum(s1_addr_len, c_addr_len) > 0
    )

    # --------------------------------------------------
    # 8. RapidFuzz — BULK / MULTI-CORE
    # --------------------------------------------------

    name_char_similarity = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    name_token_set_ratio = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.token_set_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    name_token_sort_ratio = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.token_sort_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_char_similarity = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_token_set_ratio = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.token_set_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_token_sort_ratio = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.token_sort_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    # --------------------------------------------------
    # IMPORTANT:
    # Training explicitly sets fuzzy values to 0
    # when either side is empty.
    #
    # RapidFuzz may return 100 for empty vs empty,
    # so correct those rows.
    # --------------------------------------------------

    name_invalid = (
        (s1_names == "")
        | (candidate_names == "")
    )

    address_invalid = (
        (s1_addresses == "")
        | (candidate_addresses == "")
    )

    name_char_similarity[name_invalid] = 0.0
    name_token_set_ratio[name_invalid] = 0.0
    name_token_sort_ratio[name_invalid] = 0.0

    address_char_similarity[address_invalid] = 0.0
    address_token_set_ratio[address_invalid] = 0.0
    address_token_sort_ratio[address_invalid] = 0.0

    # --------------------------------------------------
    # 9. Token overlap features
    #
    # Calculate token sets once per UNIQUE string,
    # rather than once for every candidate pair.
    # --------------------------------------------------

    name_token_cache = {
        text: set(text.split())
        for text in pd.unique(
            np.concatenate([
                s1_names,
                candidate_names
            ])
        )
        if text
    }

    address_token_cache = {
        text: set(text.split())
        for text in pd.unique(
            np.concatenate([
                s1_addresses,
                candidate_addresses
            ])
        )
        if text
    }

    name_shared_tokens = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    name_token_jaccard = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    address_shared_tokens = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    address_token_jaccard = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    for i in range(len(s1_ids)):

        # Name
        a = name_token_cache.get(
            s1_names[i],
            set()
        )

        b = name_token_cache.get(
            candidate_names[i],
            set()
        )

        intersection = len(a & b)
        union = len(a | b)

        name_shared_tokens[i] = intersection

        if union:
            name_token_jaccard[i] = (
                intersection / union
            )

        # Address
        a = address_token_cache.get(
            s1_addresses[i],
            set()
        )

        b = address_token_cache.get(
            candidate_addresses[i],
            set()
        )

        intersection = len(a & b)
        union = len(a | b)

        address_shared_tokens[i] = intersection

        if union:
            address_token_jaccard[i] = (
                intersection / union
            )

    # --------------------------------------------------
    # 10. EXACT SAME FEATURE ORDER AS TRAINING
    # --------------------------------------------------

    X = np.column_stack([
        name_exact,
        name_token_jaccard,
        name_char_similarity,
        name_token_set_ratio,
        name_token_sort_ratio,
        name_shared_tokens,
        name_length_ratio,

        address_exact,
        address_token_jaccard,
        address_char_similarity,
        address_token_set_ratio,
        address_token_sort_ratio,
        address_shared_tokens,
        address_length_ratio,

        country_match,
        name_missing,
        address_missing
    ]).astype(np.float32)

    return X, s1_ids, candidate_ids

In [ ]:
import time

benchmark_chunk = candidate_pairs.iloc[:2000]

start_time = time.time()

X_bench, bench_s1, bench_candidates = (
    build_chunk_features_fast(
        benchmark_chunk
    )
)

elapsed = time.time() - start_time

print("Feature matrix:", X_bench.shape)
print("Expected columns:", len(FEATURE_COLUMNS))
print("Candidate pairs:", len(bench_candidates))
print(f"Feature generation time: {elapsed:.2f} seconds")

In [24]:
sample_n = 100

sample_pairs = candidate_pairs.iloc[:1].copy()

row = sample_pairs.iloc[0]

s1_id = row["source1_entity_id"]

candidate_ids = [
    x.strip()
    for x in row["candidate_entity_ids"].split(",")
    if x.strip()
][:sample_n]

s1_rec = s1_cache.loc[s1_id]

old_features = []

for cid in candidate_ids:

    c = candidate_cache.loc[cid]

    old_features.append(
        build_feature_row_cached(
            s1_rec,
            c
        )
    )

old_features = np.asarray(
    old_features,
    dtype=np.float32
)

# Build equivalent mini candidate_pairs
mini = pd.DataFrame({
    "source1_entity_id": [s1_id],
    "candidate_entity_ids": [
        ",".join(candidate_ids)
    ]
})

new_features, _, _ = (
    build_chunk_features_fast(mini)
)

difference = np.max(
    np.abs(
        old_features - new_features
    )
)

print("Old shape:", old_features.shape)
print("New shape:", new_features.shape)
print("Maximum feature difference:", difference)

Old shape: (66, 17)
New shape: (66, 17)
Maximum feature difference: 5.9604645e-08


In [25]:
CHUNK_SIZE = 2_000

total_chunks = (
    len(candidate_pairs) + CHUNK_SIZE - 1
) // CHUNK_SIZE

print("Total chunks:", total_chunks)

checkpoint_files = list(
    CHECKPOINT_DIR.glob("chunk_*.parquet")
)

print(
    "Existing checkpoints:",
    len(checkpoint_files)
)

for chunk_id, start in enumerate(
    range(0, len(candidate_pairs), CHUNK_SIZE)
):

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"chunk_{chunk_id:05d}.parquet"
    )

    # -----------------------------------------
    # Resume from existing checkpoints
    # -----------------------------------------
    if checkpoint_path.exists():

        print(
            f"Skipping completed chunk "
            f"{chunk_id + 1}/{total_chunks}"
        )

        continue

    end = min(
        start + CHUNK_SIZE,
        len(candidate_pairs)
    )

    chunk = candidate_pairs.iloc[start:end]

    print(
        f"\nProcessing chunk "
        f"{chunk_id + 1}/{total_chunks} "
        f"| S1: {start:,}-{end:,}"
    )

    # -----------------------------------------
    # Build ALL 17 features in bulk
    # -----------------------------------------
    X_chunk, chunk_s1_ids, chunk_candidate_ids = (
        build_chunk_features_fast(chunk)
    )

    print(
        f"Feature matrix: {X_chunk.shape}"
    )

    # -----------------------------------------
    # XGBoost prediction
    # -----------------------------------------
    probabilities = model.predict_proba(
        X_chunk
    )[:, 1]

    # -----------------------------------------
    # Collect matches
    # -----------------------------------------
    matched_by_s1 = {}

    for (
        s1_id,
        candidate_id,
        probability
    ) in zip(
        chunk_s1_ids,
        chunk_candidate_ids,
        probabilities
    ):

        if probability >= MATCH_THRESHOLD:

            matched_by_s1.setdefault(
                s1_id,
                []
            ).append(candidate_id)

    # -----------------------------------------
    # Exactly ONE row per S1
    # -----------------------------------------
    results = []

    for row in chunk.itertuples(index=False):

        s1_id = row.source1_entity_id

        matches = matched_by_s1.get(
            s1_id,
            []
        )

        # Remove duplicates, preserve order
        matches = list(
            dict.fromkeys(matches)
        )

        results.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(
                matches
            )
        })

    checkpoint_df = pd.DataFrame(results)

    # -----------------------------------------
    # Save checkpoint
    # -----------------------------------------
    checkpoint_df.to_parquet(
        checkpoint_path,
        index=False
    )

    print(
        f"Completed chunk "
        f"{chunk_id + 1}/{total_chunks}"
        f" | candidates: "
        f"{len(chunk_candidate_ids):,}"
    )

Total chunks: 867
Existing checkpoints: 38
Skipping completed chunk 1/867
Skipping completed chunk 2/867
Skipping completed chunk 3/867
Skipping completed chunk 4/867
Skipping completed chunk 5/867
Skipping completed chunk 6/867
Skipping completed chunk 7/867
Skipping completed chunk 8/867
Skipping completed chunk 9/867
Skipping completed chunk 10/867
Skipping completed chunk 11/867
Skipping completed chunk 12/867
Skipping completed chunk 13/867
Skipping completed chunk 14/867
Skipping completed chunk 15/867
Skipping completed chunk 16/867
Skipping completed chunk 17/867
Skipping completed chunk 18/867
Skipping completed chunk 19/867
Skipping completed chunk 20/867
Skipping completed chunk 21/867
Skipping completed chunk 22/867
Skipping completed chunk 23/867
Skipping completed chunk 24/867
Skipping completed chunk 25/867
Skipping completed chunk 26/867
Skipping completed chunk 27/867
Skipping completed chunk 28/867
Skipping completed chunk 29/867
Skipping completed chunk 30/867
Skippi

In [26]:
checkpoint_files = sorted(
    CHECKPOINT_DIR.glob("chunk_*.parquet")
)

print(
    "Checkpoint files:",
    len(checkpoint_files)
)

print(
    "Expected:",
    total_chunks
)

all_results = pd.concat(
    [
        pd.read_parquet(path)
        for path in checkpoint_files
    ],
    ignore_index=True
)

print("Final rows:", len(all_results))
print(all_results.head())

Checkpoint files: 867
Expected: 867
Final rows: 1732544
  source1_entity_id                                 matched_entity_ids
0      S1-714132312                          S3-625880872,S3-867809779
1      S1-106407869                          S2-705547832,S3-613056593
2      S1-156285671             S2-172049742,S2-648730048,S3-871416040
3      S1-689823050             S2-212601842,S2-319486263,S2-829077756
4      S1-921369899  S2-808920974,S2-995426671,S3-285097097,S3-5759...


In [27]:
print(
    "Expected S1:",
    len(s1_test)
)

print(
    "Result S1:",
    len(all_results)
)

print(
    "Duplicate S1:",
    all_results["source1_entity_id"].duplicated().sum()
)

print(
    "Empty predictions:",
    (
        all_results["matched_entity_ids"]
        .fillna("")
        .eq("")
        .sum()
    )
)

Expected S1: 1732544
Result S1: 1732544
Duplicate S1: 0
Empty predictions: 208350


In [28]:
# Build candidate lookup from the official frozen candidate_pairs file

candidate_map = dict(
    zip(
        candidate_pairs["source1_entity_id"],
        candidate_pairs["candidate_entity_ids"]
    )
)

invalid_predictions = 0
duplicate_predictions = 0
prediction_count = 0

for row in all_results.itertuples(index=False):

    s1_id = row.source1_entity_id
    predicted = row.matched_entity_ids

    if pd.isna(predicted) or not predicted:
        continue

    predicted_ids = [
        x.strip()
        for x in predicted.split(",")
        if x.strip()
    ]

    # Duplicate IDs within prediction
    if len(predicted_ids) != len(set(predicted_ids)):
        duplicate_predictions += 1

    # Candidate IDs for this S1
    candidate_string = candidate_map.get(s1_id, "")

    candidate_ids = set(
        x.strip()
        for x in candidate_string.split(",")
        if x.strip()
    )

    # Every prediction must belong to candidate set
    for pid in predicted_ids:

        prediction_count += 1

        if pid not in candidate_ids:
            invalid_predictions += 1

print("Total predicted matches:", prediction_count)
print("Rows with duplicate predictions:", duplicate_predictions)
print("Invalid predictions:", invalid_predictions)

Total predicted matches: 4409432
Rows with duplicate predictions: 0
Invalid predictions: 0


In [29]:
expected_s1 = set(s1_test["entity_id"])
result_s1 = set(all_results["source1_entity_id"])

print("Missing S1:", len(expected_s1 - result_s1))
print("Extra S1:", len(result_s1 - expected_s1))

Missing S1: 0
Extra S1: 0


In [30]:
FINAL_OUTPUT = OUTPUT_DIR / "matching_results.tsv"

final_results = all_results[
    ["source1_entity_id", "matched_entity_ids"]
].copy()

final_results.to_csv(
    FINAL_OUTPUT,
    sep="\t",
    index=False
)

print("Saved:", FINAL_OUTPUT)
print("Rows:", len(final_results))

Saved: D:\V-Max_Hackerz\V-Max_Hackerz\output\matching_results.tsv
Rows: 1732544


In [31]:
check_results = pd.read_csv(
    FINAL_OUTPUT,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Shape:", check_results.shape)
print("Columns:", check_results.columns.tolist())
print("Duplicate S1:", check_results["source1_entity_id"].duplicated().sum())
print(
    "Empty predictions:",
    check_results["matched_entity_ids"].eq("").sum()
)

Shape: (1732544, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']
Duplicate S1: 0
Empty predictions: 208350


In [32]:
# Final submission sanity check

print("========== FINAL SANITY CHECK ==========")

# 1. Row count
print("Rows:", len(check_results))
print("Expected:", len(s1_test))

# 2. Columns
print("Columns:", check_results.columns.tolist())

# 3. Duplicate S1
print(
    "Duplicate S1:",
    check_results["source1_entity_id"].duplicated().sum()
)

# 4. Missing S1
expected_ids = set(s1_test["entity_id"])
result_ids = set(check_results["source1_entity_id"])

print(
    "Missing S1:",
    len(expected_ids - result_ids)
)

print(
    "Extra S1:",
    len(result_ids - expected_ids)
)

# 5. Empty predictions
print(
    "Empty predictions:",
    check_results["matched_entity_ids"]
    .eq("")
    .sum()
)

# 6. Null values
print(
    "Null source1 IDs:",
    check_results["source1_entity_id"].isna().sum()
)

print(
    "Null matched IDs:",
    check_results["matched_entity_ids"].isna().sum()
)

print("========================================")

========== FINAL SANITY CHECK ==========
Rows: 1732544
Expected: 1732544
Columns: ['source1_entity_id', 'matched_entity_ids']
Duplicate S1: 0
Missing S1: 0
Extra S1: 0
Empty predictions: 208350
Null source1 IDs: 0
Null matched IDs: 0


In [34]:
import sys

print(sys.executable)

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install lightgbm==4.7.0

D:\V-Max_Hackerz\.venv\Scripts\python.exe
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 1.1 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 1.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.1 MB/s  0:00:01


In [35]:
import lightgbm as lgb
from pathlib import Path

RESULTS_DIR = Path("../results").resolve()

LGB_MODEL_PATH = RESULTS_DIR / "lightgbm_entity_resolution.txt"
LGB_THRESHOLD_PATH = RESULTS_DIR / "lightgbm_decision_threshold.txt"

lgb_booster = lgb.Booster(
    model_file=str(LGB_MODEL_PATH)
)

LGB_THRESHOLD = float(
    LGB_THRESHOLD_PATH.read_text().strip()
)

print("LightGBM loaded successfully")
print("Threshold:", LGB_THRESHOLD)

LightGBM loaded successfully
Threshold: 0.68


In [10]:
# LightGBM inference directories

LGB_CHECKPOINT_DIR = RESULTS_DIR / "lgb_inference_checkpoints"
LGB_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LGB_OUTPUT_PATH = RESULTS_DIR / "matching_results_lgb.tsv"

print("LightGBM checkpoint directory:")
print(LGB_CHECKPOINT_DIR)

print("\nLightGBM output:")
print(LGB_OUTPUT_PATH)

LightGBM checkpoint directory:
D:\V-Max_Hackerz\results\lgb_inference_checkpoints

LightGBM output:
D:\V-Max_Hackerz\results\matching_results_lgb.tsv


In [36]:
# Generate the same 17 features
X_test_chunk, _, _ = build_chunk_features_fast(
    test_candidate_chunk
)

print("Feature shape:", X_test_chunk.shape)
print("Expected features:", len(FEATURE_COLUMNS))

# LightGBM prediction
test_probabilities = lgb_booster.predict(
    X_test_chunk
)

test_predictions = test_probabilities >= LGB_THRESHOLD

print("Predictions:", len(test_predictions))
print("Predicted matches:", test_predictions.sum())
print(
    "Probability range:",
    test_probabilities.min(),
    test_probabilities.max()
)

Feature shape: (226432, 17)
Expected features: 17
Predictions: 226432
Predicted matches: 5087
Probability range: 8.524207622951128e-09 0.9999961689579857


In [38]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# LightGBM inference configuration
# ---------------------------------------------------------

CHUNK_SIZE = 2000

LGB_CHECKPOINT_DIR = RESULTS_DIR / "lgb_inference_checkpoints"
LGB_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LGB_OUTPUT_PATH = RESULTS_DIR / "matching_results_lgb.tsv"

print("Checkpoint directory:")
print(LGB_CHECKPOINT_DIR)

print("Output:")
print(LGB_OUTPUT_PATH)


# ---------------------------------------------------------
# Process candidate_pairs in S1 chunks
# candidate_pairs has exactly one row per S1
# ---------------------------------------------------------

total_s1 = len(candidate_pairs)

num_chunks = (total_s1 + CHUNK_SIZE - 1) // CHUNK_SIZE

print("\nTotal S1:", total_s1)
print("Chunk size:", CHUNK_SIZE)
print("Expected chunks:", num_chunks)


for chunk_idx in range(num_chunks):

    checkpoint_path = (
        LGB_CHECKPOINT_DIR /
        f"chunk_{chunk_idx:05d}.parquet"
    )

    # Resume support
    if checkpoint_path.exists():
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_s1)

    candidate_pairs_chunk = candidate_pairs.iloc[start:end].copy()

    # -----------------------------------------------------
    # Generate features
    # Returns:
    #   X_features
    #   pair_s1_ids
    #   pair_candidate_ids
    # -----------------------------------------------------

    X_features, pair_s1_ids, pair_candidate_ids = (
        build_chunk_features_fast(
            candidate_pairs_chunk
        )
    )

    # -----------------------------------------------------
    # LightGBM prediction
    # -----------------------------------------------------

    if len(X_features) > 0:

        probabilities = lgb_booster.predict(
            X_features
        )

        predictions = probabilities >= LGB_THRESHOLD

    else:
        probabilities = np.array([])
        predictions = np.array([], dtype=bool)

    # -----------------------------------------------------
    # Build results for ALL S1s in this chunk
    # including S1s with zero candidates
    # -----------------------------------------------------

    chunk_s1_ids = candidate_pairs_chunk[
        "source1_entity_id"
    ].astype(str).tolist()

    matched_map = {
        s1_id: []
        for s1_id in chunk_s1_ids
    }

    # Add predicted matches
    for s1_id, candidate_id, is_match in zip(
        pair_s1_ids,
        pair_candidate_ids,
        predictions
    ):
        if is_match:
            matched_map[str(s1_id)].append(
                str(candidate_id)
            )

    # Remove accidental duplicates while preserving order
    output_rows = []

    for s1_id in chunk_s1_ids:

        matches = matched_map[s1_id]

        matches = list(dict.fromkeys(matches))

        output_rows.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(matches)
        })

    checkpoint_df = pd.DataFrame(output_rows)

    # -----------------------------------------------------
    # Save checkpoint
    # -----------------------------------------------------

    checkpoint_df.to_parquet(
        checkpoint_path,
        index=False
    )

    print(
        f"Chunk {chunk_idx + 1}/{num_chunks} "
        f"| S1: {start:,}-{end:,} "
        f"| Candidate pairs: {len(X_features):,} "
        f"| Predicted matches: {int(predictions.sum()):,}"
    )


print("\nLightGBM inference complete.")

Checkpoint directory:
D:\V-Max_Hackerz\results\lgb_inference_checkpoints
Output:
D:\V-Max_Hackerz\results\matching_results_lgb.tsv

Total S1: 1732544
Chunk size: 2000
Expected chunks: 867
Chunk 1/867 | S1: 0-2,000 | Candidate pairs: 226,432 | Predicted matches: 5,087
Chunk 2/867 | S1: 2,000-4,000 | Candidate pairs: 234,722 | Predicted matches: 5,204
Chunk 3/867 | S1: 4,000-6,000 | Candidate pairs: 234,218 | Predicted matches: 5,088
Chunk 4/867 | S1: 6,000-8,000 | Candidate pairs: 233,725 | Predicted matches: 5,168
Chunk 5/867 | S1: 8,000-10,000 | Candidate pairs: 241,711 | Predicted matches: 5,163
Chunk 6/867 | S1: 10,000-12,000 | Candidate pairs: 230,562 | Predicted matches: 5,160
Chunk 7/867 | S1: 12,000-14,000 | Candidate pairs: 238,638 | Predicted matches: 5,109
Chunk 8/867 | S1: 14,000-16,000 | Candidate pairs: 242,758 | Predicted matches: 5,344
Chunk 9/867 | S1: 16,000-18,000 | Candidate pairs: 245,436 | Predicted matches: 5,273
Chunk 10/867 | S1: 18,000-20,000 | Candidate pairs:

KeyboardInterrupt: 

In [39]:
# Continue LightGBM inference from existing 23 checkpoints
# Existing:
# chunk_00000 -> chunk_00022
# Covers S1 rows 0 -> 46,000

CHUNK_SIZE = 10000
START_ROW = 23 * 2000          # 46,000
START_CHECKPOINT = 23

remaining_rows = len(candidate_pairs) - START_ROW
remaining_chunks = (
    remaining_rows + CHUNK_SIZE - 1
) // CHUNK_SIZE

print("Already completed S1 rows:", START_ROW)
print("Remaining S1 rows:", remaining_rows)
print("New chunk size:", CHUNK_SIZE)
print("New chunks:", remaining_chunks)

Already completed S1 rows: 46000
Remaining S1 rows: 1686544
New chunk size: 10000
New chunks: 169


In [40]:
for local_idx in range(remaining_chunks):

    chunk_idx = START_CHECKPOINT + local_idx

    start = START_ROW + local_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(candidate_pairs))

    checkpoint_path = (
        LGB_CHECKPOINT_DIR /
        f"chunk_{chunk_idx:05d}.parquet"
    )

    if checkpoint_path.exists():
        print(f"Skipping existing chunk {chunk_idx}")
        continue

    candidate_pairs_chunk = candidate_pairs.iloc[start:end].copy()

    # Generate features
    X_features, pair_s1_ids, pair_candidate_ids = (
        build_chunk_features_fast(
            candidate_pairs_chunk
        )
    )

    # LightGBM prediction
    if len(X_features) > 0:

        probabilities = lgb_booster.predict(
            X_features
        )

        predictions = probabilities >= LGB_THRESHOLD

    else:
        probabilities = np.array([])
        predictions = np.array([], dtype=bool)

    # Create map for every S1 in this chunk
    chunk_s1_ids = (
        candidate_pairs_chunk["source1_entity_id"]
        .astype(str)
        .tolist()
    )

    matched_map = {
        s1_id: []
        for s1_id in chunk_s1_ids
    }

    # Store predicted matches
    for s1_id, candidate_id, is_match in zip(
        pair_s1_ids,
        pair_candidate_ids,
        predictions
    ):
        if is_match:
            matched_map[str(s1_id)].append(
                str(candidate_id)
            )

    # Build output rows
    output_rows = []

    for s1_id in chunk_s1_ids:

        matches = list(
            dict.fromkeys(
                matched_map[s1_id]
            )
        )

        output_rows.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(matches)
        })

    checkpoint_df = pd.DataFrame(output_rows)

    checkpoint_df.to_parquet(
        checkpoint_path,
        index=False
    )

    print(
        f"Chunk {chunk_idx + 1} | "
        f"S1: {start:,}-{end:,} | "
        f"Candidate pairs: {len(X_features):,} | "
        f"Predicted matches: {int(predictions.sum()):,}"
    )

Chunk 24 | S1: 46,000-56,000 | Candidate pairs: 1,191,750 | Predicted matches: 26,032
Chunk 25 | S1: 56,000-66,000 | Candidate pairs: 1,207,057 | Predicted matches: 26,032
Chunk 26 | S1: 66,000-76,000 | Candidate pairs: 1,184,462 | Predicted matches: 26,175


KeyboardInterrupt: 

In [41]:
xgb_results = pd.read_csv(
    "../output/matching_results.tsv",
    sep="\t",
    dtype=str
)

print("XGB rows:", len(xgb_results))
print("XGB empty:", (xgb_results["matched_entity_ids"] == "").sum())

XGB rows: 1732544
XGB empty: 0


In [42]:
xgb_positive_pairs = (
    xgb_results[
        xgb_results["matched_entity_ids"].notna() &
        (xgb_results["matched_entity_ids"] != "")
    ][
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        candidate_entity_id=lambda df:
            df["matched_entity_ids"].str.split(",")
    )
    .explode("candidate_entity_id")
    [["source1_entity_id", "candidate_entity_id"]]
    .reset_index(drop=True)
)

xgb_positive_pairs["source1_entity_id"] = (
    xgb_positive_pairs["source1_entity_id"].astype(str)
)

xgb_positive_pairs["candidate_entity_id"] = (
    xgb_positive_pairs["candidate_entity_id"].astype(str)
)

print("XGB positive pairs:", len(xgb_positive_pairs))
print(
    "Unique S1:",
    xgb_positive_pairs["source1_entity_id"].nunique()
)

XGB positive pairs: 4409432
Unique S1: 1524194


In [43]:
lgb_rerank_pairs = xgb_positive_pairs.rename(
    columns={
        "candidate_entity_id": "candidate_entity_ids"
    }
).copy()

print(lgb_rerank_pairs.head())
print("Pairs:", len(lgb_rerank_pairs))

  source1_entity_id candidate_entity_ids
0      S1-714132312         S3-625880872
1      S1-714132312         S3-867809779
2      S1-106407869         S2-705547832
3      S1-106407869         S3-613056593
4      S1-156285671         S2-172049742
Pairs: 4409432


In [44]:
test_rerank = lgb_rerank_pairs.iloc[:10000].copy()

X_rerank, rerank_s1, rerank_candidate = (
    build_chunk_features_fast(test_rerank)
)

print("Feature shape:", X_rerank.shape)

rerank_prob = lgb_booster.predict(X_rerank)

rerank_pred = rerank_prob >= LGB_THRESHOLD

print("Predictions:", len(rerank_pred))
print("LightGBM retained:", rerank_pred.sum())
print(
    "Probability range:",
    rerank_prob.min(),
    rerank_prob.max()
)

Feature shape: (10000, 17)
Predictions: 10000
LightGBM retained: 9673
Probability range: 0.14808391771295898 0.9999961689579857


In [45]:
LGB_RERANK_DIR = RESULTS_DIR / "lgb_rerank_checkpoints"
LGB_RERANK_DIR.mkdir(parents=True, exist_ok=True)

RERANK_CHUNK_SIZE = 50000

total_pairs = len(lgb_rerank_pairs)
num_rerank_chunks = (
    total_pairs + RERANK_CHUNK_SIZE - 1
) // RERANK_CHUNK_SIZE

print("Total XGB-positive pairs:", total_pairs)
print("Chunk size:", RERANK_CHUNK_SIZE)
print("Expected chunks:", num_rerank_chunks)

Total XGB-positive pairs: 4409432
Chunk size: 50000
Expected chunks: 89


In [46]:
for chunk_idx in range(num_rerank_chunks):

    checkpoint_path = (
        LGB_RERANK_DIR /
        f"chunk_{chunk_idx:05d}.parquet"
    )

    # Resume support
    if checkpoint_path.exists():
        continue

    start = chunk_idx * RERANK_CHUNK_SIZE
    end = min(
        start + RERANK_CHUNK_SIZE,
        total_pairs
    )

    chunk = lgb_rerank_pairs.iloc[start:end].copy()

    # Generate 17 features
    X_chunk, pair_s1_ids, pair_candidate_ids = (
        build_chunk_features_fast(chunk)
    )

    # LightGBM reranking
    probabilities = lgb_booster.predict(X_chunk)

    predictions = probabilities >= LGB_THRESHOLD

    # Keep only pairs accepted by LightGBM
    kept_s1 = np.asarray(pair_s1_ids)[predictions]
    kept_candidate = np.asarray(pair_candidate_ids)[predictions]
    kept_prob = probabilities[predictions]

    checkpoint_df = pd.DataFrame({
        "source1_entity_id": kept_s1.astype(str),
        "candidate_entity_id": kept_candidate.astype(str),
        "lgb_probability": kept_prob
    })

    checkpoint_df.to_parquet(
        checkpoint_path,
        index=False
    )

    print(
        f"Chunk {chunk_idx + 1}/{num_rerank_chunks} | "
        f"Pairs: {start:,}-{end:,} | "
        f"Input: {len(X_chunk):,} | "
        f"Retained: {len(checkpoint_df):,}"
    )

Chunk 1/89 | Pairs: 0-50,000 | Input: 50,000 | Retained: 48,346
Chunk 2/89 | Pairs: 50,000-100,000 | Input: 50,000 | Retained: 48,247
Chunk 3/89 | Pairs: 100,000-150,000 | Input: 50,000 | Retained: 48,338
Chunk 4/89 | Pairs: 150,000-200,000 | Input: 50,000 | Retained: 48,337
Chunk 5/89 | Pairs: 200,000-250,000 | Input: 50,000 | Retained: 48,365
Chunk 6/89 | Pairs: 250,000-300,000 | Input: 50,000 | Retained: 48,299
Chunk 7/89 | Pairs: 300,000-350,000 | Input: 50,000 | Retained: 48,337
Chunk 8/89 | Pairs: 350,000-400,000 | Input: 50,000 | Retained: 48,416
Chunk 9/89 | Pairs: 400,000-450,000 | Input: 50,000 | Retained: 48,389
Chunk 10/89 | Pairs: 450,000-500,000 | Input: 50,000 | Retained: 48,355
Chunk 11/89 | Pairs: 500,000-550,000 | Input: 50,000 | Retained: 48,322
Chunk 12/89 | Pairs: 550,000-600,000 | Input: 50,000 | Retained: 48,346
Chunk 13/89 | Pairs: 600,000-650,000 | Input: 50,000 | Retained: 48,365
Chunk 14/89 | Pairs: 650,000-700,000 | Input: 50,000 | Retained: 48,312
Chunk 15/

In [47]:
# Combine LightGBM reranker checkpoints

import pandas as pd
from pathlib import Path

rerank_files = sorted(
    LGB_RERANK_DIR.glob("chunk_*.parquet")
)

print("Checkpoint files:", len(rerank_files))
print("Expected:", num_rerank_chunks)

rerank_parts = [
    pd.read_parquet(path)
    for path in rerank_files
]

lgb_reranked_pairs = pd.concat(
    rerank_parts,
    ignore_index=True
)

print("\nTotal XGB-positive pairs:", len(lgb_rerank_pairs))
print("LightGBM-retained pairs:", len(lgb_reranked_pairs))

print(
    "Retention:",
    len(lgb_reranked_pairs) / len(lgb_rerank_pairs)
)

print(
    "Removed:",
    len(lgb_rerank_pairs) - len(lgb_reranked_pairs)
)

print("\nDuplicate pairs:",
      lgb_reranked_pairs.duplicated(
          ["source1_entity_id", "candidate_entity_id"]
      ).sum())

print("\nSample:")
display(lgb_reranked_pairs.head())

Checkpoint files: 89
Expected: 89

Total XGB-positive pairs: 4409432
LightGBM-retained pairs: 4262476
Retention: 0.9666723514502548
Removed: 146956

Duplicate pairs: 0

Sample:


,source1_entity_id,candidate_entity_id,lgb_probability
0,S1-714132312,S3-625880872,0.998688
1,S1-714132312,S3-867809779,0.976911
2,S1-106407869,S2-705547832,0.960914
3,S1-106407869,S3-613056593,0.973223
4,S1-156285671,S2-172049742,0.999205


In [48]:
# Build hybrid XGBoost → LightGBM result

# Start from original XGBoost result
hybrid_results = xgb_results[
    ["source1_entity_id"]
].copy()

# Group LightGBM-retained pairs by S1
lgb_matches = (
    lgb_reranked_pairs
    .groupby("source1_entity_id")["candidate_entity_id"]
    .agg(lambda x: ",".join(dict.fromkeys(x.astype(str))))
    .rename("matched_entity_ids")
)

# Attach LightGBM-filtered matches
hybrid_results = hybrid_results.merge(
    lgb_matches,
    left_on="source1_entity_id",
    right_index=True,
    how="left"
)

# S1s with no retained matches get empty string
hybrid_results["matched_entity_ids"] = (
    hybrid_results["matched_entity_ids"]
    .fillna("")
)

# Save separately
HYBRID_OUTPUT_PATH = (
    RESULTS_DIR / "matching_results_hybrid.tsv"
)

hybrid_results.to_csv(
    HYBRID_OUTPUT_PATH,
    sep="\t",
    index=False
)

print("Saved:", HYBRID_OUTPUT_PATH)
print("Shape:", hybrid_results.shape)
print(
    "Empty predictions:",
    (hybrid_results["matched_entity_ids"] == "").sum()
)
print(
    "Duplicate S1:",
    hybrid_results["source1_entity_id"].duplicated().sum()
)

Saved: D:\V-Max_Hackerz\results\matching_results_hybrid.tsv
Shape: (1732544, 2)
Empty predictions: 223942
Duplicate S1: 0


In [49]:
# Compare XGBoost vs hybrid at S1 level

xgb_nonempty = set(
    xgb_results.loc[
        xgb_results["matched_entity_ids"].fillna("") != "",
        "source1_entity_id"
    ].astype(str)
)

hybrid_nonempty = set(
    hybrid_results.loc[
        hybrid_results["matched_entity_ids"].fillna("") != "",
        "source1_entity_id"
    ].astype(str)
)

created_matches = hybrid_nonempty - xgb_nonempty
lost_matches = xgb_nonempty - hybrid_nonempty

print("XGB non-empty S1:", len(xgb_nonempty))
print("Hybrid non-empty S1:", len(hybrid_nonempty))

print("\nS1s that hybrid CREATED matches for:", len(created_matches))
print("S1s that hybrid LOST all matches:", len(lost_matches))

if created_matches:
    print("\nWARNING: Unexpected created matches!")
    print("Sample:", list(created_matches)[:10])

XGB non-empty S1: 1524194
Hybrid non-empty S1: 1508602

S1s that hybrid CREATED matches for: 0
S1s that hybrid LOST all matches: 15592


In [50]:
print("XGB pairs:", len(xgb_positive_pairs))
print("Reranker input pairs:", len(lgb_rerank_pairs))
print("Reranker retained:", len(lgb_reranked_pairs))

XGB pairs: 4409432
Reranker input pairs: 4409432
Reranker retained: 4262476
